In [1]:
print("=== Floating Point Precision ===")
print(f"0.1 + 0.2 = {0.1 + 0.2}")
print(f"0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")
print(f"Difference: {(0.1 + 0.2) - 0.3:.2e}")

=== Floating Point Precision ===
0.1 + 0.2 = 0.30000000000000004
0.1 + 0.2 == 0.3? False
Difference: 5.55e-17


Implement naive vs stable softmax

In [4]:
import math

def softmax_naive(logits):
    exps = [math.exp(z) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]

def softmax_stable(logits):
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]

safe_logits = [2.0, 1.0, 0.1]
print(f"Naive:  {softmax_naive(safe_logits)}")
print(f"Stable: {softmax_stable(safe_logits)}")

dangerous_logits = [100.0, 101.0, 102.0]
print(f"Naive:  {softmax_naive(dangerous_logits)}")
print(f"Stable: {softmax_stable(dangerous_logits)}")


Naive:  [0.6590011388859679, 0.2424329707047139, 0.09856589040931818]
Stable: [0.6590011388859679, 0.24243297070471392, 0.09856589040931818]
Naive:  [0.09003057317038046, 0.24472847105479764, 0.6652409557748219]
Stable: [0.09003057317038046, 0.24472847105479764, 0.6652409557748218]


Implement stable log-sum-exp

In [5]:
def logsumexp_naive(values):
    return math.log(sum(math.exp(v) for v in values))

def logsumexp_stable(values):
    c = max(values)
    return c + math.log(sum(math.exp(v - c) for v in values))

safe = [1.0, 2.0, 3.0]
print(f"Naive:  {logsumexp_naive(safe):.6f}")
print(f"Stable: {logsumexp_stable(safe):.6f}")

large = [500.0, 501.0, 502.0]
print(f"Naive:  {logsumexp_naive(large):.6f}")
print(f"Stable: {logsumexp_stable(large):.6f}")


Naive:  3.407606
Stable: 3.407606
Naive:  502.407606
Stable: 502.407606


Implement stable cross-entropy

In [6]:
def cross_entropy_naive(true_class, logits):
    probs = softmax_naive(logits)
    return -math.log(probs[true_class])

def cross_entropy_stable(true_class, logits):
    max_logit = max(logits)
    shifted = [z - max_logit for z in logits]
    log_sum_exp = math.log(sum(math.exp(s) for s in shifted))
    log_prob = shifted[true_class] - log_sum_exp
    return -log_prob

logits = [2.0, 5.0, 1.0]
true_class = 1
print(f"Naive:  {cross_entropy_naive(true_class, logits):.6f}")
print(f"Stable: {cross_entropy_stable(true_class, logits):.6f}")

Naive:  0.065884
Stable: 0.065884


Gradient checking

In [7]:
def numerical_gradient(f, x, h=1e-5):
    grad = []
    for i in range(len(x)):
        x_plus = x[:]
        x_minus = x[:]
        x_plus[i] += h
        x_minus[i] -= h
        grad.append((f(x_plus) - f(x_minus)) / (2 * h))
    return grad

def check_gradient(analytical, numerical, tolerance=1e-5):
    for i, (a, n) in enumerate(zip(analytical, numerical)):
        denom = max(abs(a), abs(n), 1e-8)
        rel_error = abs(a - n) / denom
        status = "OK" if rel_error < tolerance else "FAIL"
        print(f"  param {i}: analytical={a:.8f} numerical={n:.8f} "
              f"rel_error={rel_error:.2e} [{status}]")

def f(params):
    x, y = params
    return x**2 + 3*x*y + y**3

def f_grad(params):
    x, y = params
    return [2*x + 3*y, 3*x + 3*y**2]

point = [2.0, 1.0]
analytical = f_grad(point)
numerical = numerical_gradient(f, point)
check_gradient(analytical, numerical)

  param 0: analytical=7.00000000 numerical=7.00000000 rel_error=1.29e-11 [OK]
  param 1: analytical=9.00000000 numerical=9.00000000 rel_error=1.15e-11 [OK]


Mixed precision simulation

In [9]:
import struct

def float32_to_float16_round(x):
    packed = struct.pack('f', x)
    f32 = struct.unpack('f', packed)[0]
    packed16 = struct.pack('e', f32)
    return struct.unpack('e', packed16)[0]

def simulate_bfloat16(x):
    packed = struct.pack('f', x)
    as_int = int.from_bytes(packed, 'little')
    truncated = as_int & 0xFFFF0000
    repacked = truncated.to_bytes(4, 'little')
    return struct.unpack('f', repacked)[0]

def check(label, x):
    f16 = float32_to_float16_round(x)
    bf16 = simulate_bfloat16(x)
    print(f"{label:35s} in={x!r:>22}  f16={f16!r:>22}  bf16={bf16!r:>22}")

In [13]:
print("--- exact values ---")
check("zero",              0.0)
check("negative zero",    -0.0)
check("one",               1.0)
check("minus one",        -1.0)
check("two",               2.0)
check("half",              0.5)
check("quarter",           0.25)
check("power of two",      1024.0)
check("large power of 2",  32768.0)   # 2^15, exactly representable in f16

--- exact values ---
zero                                in=                   0.0  f16=                   0.0  bf16=                   0.0
negative zero                       in=                  -0.0  f16=                  -0.0  bf16=                  -0.0
one                                 in=                   1.0  f16=                   1.0  bf16=                   1.0
minus one                           in=                  -1.0  f16=                  -1.0  bf16=                  -1.0
two                                 in=                   2.0  f16=                   2.0  bf16=                   2.0
half                                in=                   0.5  f16=                   0.5  bf16=                   0.5
quarter                             in=                  0.25  f16=                  0.25  bf16=                  0.25
power of two                        in=                1024.0  f16=                1024.0  bf16=                1024.0
large power of 2           

Gradient clipping

In [14]:
def clip_by_norm(gradients, max_norm):
    total_norm = math.sqrt(sum(g**2 for g in gradients))
    if total_norm > max_norm:
        scale = max_norm / total_norm
        return [g * scale for g in gradients]
    return gradients

grads = [10.0, 20.0, 30.0]
clipped = clip_by_norm(grads, max_norm=5.0)
print(f"Original norm: {math.sqrt(sum(g**2 for g in grads)):.2f}")
print(f"Clipped norm:  {math.sqrt(sum(g**2 for g in clipped)):.2f}")
print(f"Direction preserved: {[c/clipped[0] for c in clipped]} == {[g/grads[0] for g in grads]}")

Original norm: 37.42
Clipped norm:  5.00
Direction preserved: [1.0, 2.0, 3.0] == [1.0, 2.0, 3.0]


NaN/Inf detection

In [15]:
def check_tensor(name, values):
    has_nan = any(math.isnan(v) for v in values)
    has_inf = any(math.isinf(v) for v in values)
    if has_nan or has_inf:
        print(f"WARNING {name}: nan={has_nan} inf={has_inf}")
        return False
    return True

check_tensor("good", [1.0, 2.0, 3.0])
check_tensor("bad",  [1.0, float('nan'), 3.0])
check_tensor("ugly", [1.0, float('inf'), 3.0])

WARNING bad: nan=True inf=False
WARNING ugly: nan=False inf=True


False